# 12 — Sequence models and time-series forecasting

**Level:** Intermediate → advanced · **Study time:** 75–100 minutes (including experiments)

**Prerequisites:** Notebooks 05–08.

## What you will be able to do
- Build sliding input windows without future leakage.
- Train a GRU for next-step prediction.
- Compare with a persistence baseline.

## How to work through this lab
Run cells in order with **Shift+Enter**. Before each result, predict its shape or behavior.
Change one thing, rerun, and explain what changed. Use **Restart Kernel and Run All**
when you want a clean experiment. Lessons use local data and do not depend on earlier kernel state.
Record your observations in the experiment journal near the end.


[Course guide](../README.md) · [Previous: 11](11_embeddings_and_text.ipynb) · [Next: 13](13_attention_and_transformers.ipynb)


## Step 1 — Prepare a reproducible experiment

A seed makes comparisons easier. Exact numerical results can still differ across devices and PyTorch versions.


In [ ]:
import random
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
import torch
from torch import nn
from torch.utils.data import DataLoader, TensorDataset

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.set_num_threads(2)  # Small examples often run faster with fewer CPU threads.
device = torch.device("cpu")  # Learn on CPU first; the performance lab covers CUDA.
plt.rcParams.update({"figure.figsize": (7, 4), "axes.grid": True})
print("PyTorch:", torch.__version__, "| Device:", device)


## Step 2 — Create chronological windows

For a sequence, a random window split can place almost identical neighboring observations on both sides. We use chronological boundaries. A target at time t may use only observations before t. Validation windows may use earlier training history: that is available at prediction time.


In [ ]:
t = torch.arange(900, dtype=torch.float32)
series = torch.sin(t * 0.08) + 0.35 * torch.sin(t * 0.021) + 0.04 * torch.randn(900)
window = 24
X = torch.stack([series[i-window:i] for i in range(window, len(series))]).unsqueeze(-1)
y = series[window:].unsqueeze(1)
target_times = torch.arange(window, len(series))
tr = target_times < 600
va = (target_times >= 600) & (target_times < 750)
te = target_times >= 750
mean, std = series[:600].mean(), series[:600].std()
Xn, yn = (X - mean) / std, (y - mean) / std
loader = DataLoader(TensorDataset(Xn[tr], yn[tr]), batch_size=64, shuffle=True)
print("[batch, time, features]:", Xn.shape)
assert target_times[tr].max() < target_times[va].min()


## Step 3 — Use a recurrent hidden state

A GRU reads one time step at a time and updates a hidden representation. With `batch_first=True`, its input is `[B,T,F]`. We reset hidden state for each independent window by omitting an initial state. The final output summarizes the window.


In [ ]:
class Forecaster(nn.Module):
    def __init__(self):
        super().__init__()
        self.gru = nn.GRU(input_size=1, hidden_size=24, batch_first=True)
        self.head = nn.Linear(24, 1)
    def forward(self, x):
        output, hidden = self.gru(x)
        return self.head(output[:, -1])
model = Forecaster()
assert model(Xn[:8]).shape == (8, 1)


## Step 4 — Select a checkpoint on the next time period

Shuffling training windows is acceptable here because each window contains its own ordered history and hidden state is reset. Keep the order within each window. Choose the checkpoint using validation, then evaluate the final period.


In [ ]:
import copy
optimizer = torch.optim.Adam(model.parameters(), lr=0.008)
best_val = float('inf')
for epoch in range(25):
    model.train()
    for xb, yb in loader:
        optimizer.zero_grad(set_to_none=True)
        loss = nn.functional.mse_loss(model(xb), yb)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
    model.eval()
    with torch.no_grad():
        val_loss = nn.functional.mse_loss(model(Xn[va]), yn[va]).item()
    if val_loss < best_val:
        best_val, best_state = val_loss, copy.deepcopy(model.state_dict())
model.load_state_dict(best_state)
with torch.no_grad():
    predicted = model(Xn[te]) * std + mean
    mse = nn.functional.mse_loss(predicted, y[te]).item()
    persistence_mse = nn.functional.mse_loss(X[te, -1], y[te]).item()
print({"GRU_test_MSE": mse, "persistence_test_MSE": persistence_mse})
plt.plot(target_times[te], y[te, 0], label="actual")
plt.plot(target_times[te], predicted[:, 0], label="one-step forecast")
plt.legend(); plt.show()
assert np.isfinite(mse)


## Your turn — practice before opening the solutions

1. Compute MAE for both models.
2. Verify a sample's last input corresponds to the time immediately before its target.
3. Forecast 20 steps recursively from the beginning of the test period; compare this with one-step evaluation. Hint: feed each prediction back into the window.

Use the empty cell below for your attempt. The worked examples above run as supplied;
you do not need to fill in exercises to use **Run All**. Solutions are below the journal
and their code is initially collapsed in JupyterLab.


In [ ]:
# Your experiment goes here. Add more cells if you need them.


## Experiment journal

| Question | Your notes |
|---|---|
| What did I predict? | |
| What single setting did I change? | |
| What result did I measure? | |
| Why might it have changed? | |
| What would I try next? | |

## Reference solutions
Try the exercises first. Expand the following cell to compare your reasoning.


In [ ]:
print("GRU MAE:", (predicted - y[te]).abs().mean().item())
print("Persistence MAE:", (X[te, -1] - y[te]).abs().mean().item())
i = 100
assert X[i, -1, 0] == series[target_times[i] - 1]
context = Xn[te][0:1].clone()
recursive = []
with torch.no_grad():
    for _ in range(20):
        next_value = model(context)
        recursive.append((next_value * std + mean).item())
        context = torch.cat([context[:, 1:], next_value.unsqueeze(1)], dim=1)
plt.plot(y[te][:20, 0], label="actual")
plt.plot(recursive, label="recursive forecast"); plt.legend(); plt.show()
# Recursive evaluation accumulates model errors instead of receiving each new true observation.


## Before moving on

One-step forecasts use newly observed true history each time; recursive forecasts do not. Report which setting you evaluated. A neural network need not beat persistence on every series.

**Explain it aloud:** What went into the model or operation, what came out, and how did you verify it?

## Official references and further reading
- [GRU](https://docs.pytorch.org/docs/stable/generated/torch.nn.GRU.html)


[Course guide](../README.md) · [Previous: 11](11_embeddings_and_text.ipynb) · [Next: 13](13_attention_and_transformers.ipynb)
